# 01 — Mathematical Foundations of Logistic Regression

**Project:** The Geometry of Learning, Investigating the Effects of Loss Landscape Conditioning on Optimization Algorithms in Machine Learning

## Purpose

This notebook establishes and verifies the mathematical objects used throughout the project. We define the binary logistic-regression objective, derive its gradient and Hessian, and compare the analytical derivatives implemented in `src/logistic_regression.py` with finite-difference approximations.

## Learning objectives

- Express logistic regression as a differentiable optimization problem.
- Understand how the gradient describes the local direction of change.
- Understand how the Hessian describes local curvature.
- Verify that the shared implementation agrees with numerical derivatives.

**Convention:** the first parameter is the intercept. L2 regularization is applied to feature weights but not to the intercept.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start=None):
    """Find the repository root from the current Jupyter working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "logistic_regression.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Open Jupyter from the repository "
        "folder or one of its subfolders."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import logistic_regression, optimizers

FIGURES_PATH = PROJECT_ROOT / "results" / "figures"
TABLES_PATH = PROJECT_ROOT / "results" / "tables"
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
TABLES_PATH.mkdir(parents=True, exist_ok=True)


## 1. Model and Objective

For an observation with feature vector \(x_i\), logistic regression predicts:

$$
p_i = \sigma(x_i^\top \theta)
$$

where the sigmoid function is defined as:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

For \(m\) observations, the regularized mean binary cross-entropy loss is:

$$
J(\theta) =
-\frac{1}{m}\sum_{i=1}^{m}
\left[
y_i\log(p_i) +
(1-y_i)\log(1-p_i)
\right]
+
\frac{\lambda}{2}\sum_{j=1}^{n}\theta_j^2
$$

Here:

- \(m\) is the number of observations.
- \(n\) is the number of features.
- (λ) is the L2 regularization coefficient.
- (θ₀) represents the intercept and is excluded from regularization.

The implementation uses `numpy.logaddexp` to calculate the binary cross-entropy in a numerically stable manner.


In [2]:
# A small deterministic synthetic binary-classification dataset.
rng = np.random.default_rng(42)
X = rng.normal(size=(400, 3))
true_intercept = -0.2
true_weights = np.array([1.2, -1.5, 0.8])
true_probability = logistic_regression.sigmoid(
    true_intercept + X @ true_weights
)
y = (rng.random(X.shape[0]) < true_probability).astype(int)

X_bias = logistic_regression.add_intercept(X)
theta = np.array([0.15, -0.35, 0.25, 0.10], dtype=float)
L2 = 0.05

print("Feature matrix:", X.shape)
print("Intercept-augmented matrix:", X_bias.shape)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Objective value:", logistic_regression.binary_cross_entropy(
    X_bias, y, theta, l2=L2
))

Feature matrix: (400, 3)
Intercept-augmented matrix: (400, 4)
Class counts: {np.int64(0): np.int64(213), np.int64(1): np.int64(187)}
Objective value: 0.8343331136675574


## 2. Gradient

The gradient of the data term is
$$
\nabla J(\theta)=\frac{1}{m}X^T(p-y)+\lambda\,R\theta,
$$
where $$R=\operatorname{diag}(0,1,\ldots,1)$$ leaves the intercept unregularized.

The gradient is a vector of first-order partial derivatives. It identifies the local direction of steepest increase; moving in the negative-gradient direction is the basic gradient-descent update.

In [3]:
def central_difference_gradient(function, point, epsilon=1e-6):
    """Numerically approximate a gradient using central differences."""
    point = np.asarray(point, dtype=float)
    approximation = np.zeros_like(point)
    for j in range(point.size):
        plus = point.copy()
        minus = point.copy()
        plus[j] += epsilon
        minus[j] -= epsilon
        approximation[j] = (function(plus) - function(minus)) / (2 * epsilon)
    return approximation


objective = lambda candidate: logistic_regression.binary_cross_entropy(
    X_bias, y, candidate, l2=L2
)
analytical_gradient = logistic_regression.compute_gradient(
    X_bias, y, theta, l2=L2
)
numerical_gradient = central_difference_gradient(objective, theta)
gradient_error = np.max(np.abs(analytical_gradient - numerical_gradient))

print("Analytical gradient:", analytical_gradient)
print("Finite-difference gradient:", numerical_gradient)
print(f"Maximum absolute gradient error: {gradient_error:.3e}")
assert np.allclose(analytical_gradient, numerical_gradient, atol=1e-6, rtol=1e-5)

Analytical gradient: [ 0.06488083 -0.29802494  0.25415847 -0.07439129]
Finite-difference gradient: [ 0.06488083 -0.29802494  0.25415847 -0.07439129]
Maximum absolute gradient error: 1.139e-10


## 3. Hessian and local curvature

The Hessian of the regularized objective is

$$
 H(\theta)=\frac{1}{m}X^TWX+\lambda R,
 \qquad W_{ii}=p_i(1-p_i).
$$

The Hessian is a square matrix of second-order partial derivatives. For logistic regression it is positive semidefinite; its eigenvalues describe curvature along corresponding eigenvector directions. The ratio of the largest to smallest positive eigenvalue is commonly used to quantify local conditioning.

In [4]:
def central_difference_hessian_from_gradient(gradient_function, point, epsilon=1e-5):
    """Approximate a Hessian by finite-differencing an analytical gradient."""
    point = np.asarray(point, dtype=float)
    n_parameters = point.size
    approximation = np.zeros((n_parameters, n_parameters), dtype=float)
    for j in range(n_parameters):
        plus = point.copy()
        minus = point.copy()
        plus[j] += epsilon
        minus[j] -= epsilon
        approximation[:, j] = (
            gradient_function(plus) - gradient_function(minus)
        ) / (2 * epsilon)
    return approximation


gradient_function = lambda candidate: logistic_regression.compute_gradient(
    X_bias, y, candidate, l2=L2
)
analytical_hessian = logistic_regression.compute_hessian(
    X_bias, theta, l2=L2
)
numerical_hessian = central_difference_hessian_from_gradient(
    gradient_function, theta
)
hessian_error = np.max(np.abs(analytical_hessian - numerical_hessian))
eigenvalues = np.linalg.eigvalsh(analytical_hessian)

print("Analytical Hessian:\n", analytical_hessian)
print("Numerical Hessian:\n", numerical_hessian)
print(f"Maximum absolute Hessian error: {hessian_error:.3e}")
print("Hessian eigenvalues:", eigenvalues)
print("Hessian condition number:", np.linalg.cond(analytical_hessian))

assert np.allclose(analytical_hessian, numerical_hessian, atol=1e-5, rtol=1e-4)
assert np.allclose(analytical_hessian, analytical_hessian.T, atol=1e-12)
assert eigenvalues.min() >= -1e-10

Analytical Hessian:
 [[ 0.23821156  0.00543327 -0.00997453 -0.00892058]
 [ 0.00543327  0.29080839  0.0236726   0.00183385]
 [-0.00997453  0.0236726   0.26745191  0.00748301]
 [-0.00892058  0.00183385  0.00748301  0.26462973]]
Numerical Hessian:
 [[ 0.23821156  0.00543327 -0.00997453 -0.00892058]
 [ 0.00543327  0.29080839  0.0236726   0.00183385]
 [-0.00997453  0.0236726   0.26745191  0.00748301]
 [-0.00892058  0.00183385  0.00748301  0.26462973]]
Maximum absolute Hessian error: 1.204e-11
Hessian eigenvalues: [0.23110853 0.25261609 0.27104992 0.30632706]
Hessian condition number: 1.3254684072997864


## 4. Verification summary

The assertions above check that the implemented gradient and Hessian agree with independent finite-difference approximations, that the Hessian is symmetric, and that its eigenvalues are non-negative up to numerical tolerance.

These checks validate the local mathematical implementation on a deterministic test dataset. They do not, by themselves, establish convergence or generalization performance; those are examined in the later notebooks.

In [5]:
verification_summary = pd.DataFrame(
    {
        "Check": [
            "Gradient finite-difference agreement",
            "Hessian finite-difference agreement",
            "Hessian symmetry",
            "Positive-semidefinite Hessian (numerical tolerance)",
        ],
        "Result": [
            gradient_error,
            hessian_error,
            np.max(np.abs(analytical_hessian - analytical_hessian.T)),
            eigenvalues.min(),
        ],
    }
)
display(verification_summary)
print("All mathematical verification assertions passed.")

,Check,Result
0,Gradient finite-difference agreement,1.139148e-10
1,Hessian finite-difference agreement,1.204431e-11
2,Hessian symmetry,5.204170e-18
3,Positive-semidefinite Hessian (numerical toler...,2.311085e-01


All mathematical verification assertions passed.
